In [ ]:
import os, shutil
from pathlib import Path

def drive_is_mounted():
    try:
        return any(" /content/drive " in line for line in open("/proc/mounts"))
    except Exception:
        return os.path.ismount("/content/drive")

if drive_is_mounted():
    print("Drive already mounted — kuch delete nahi kiya.")
else:
    p = Path("/content/drive")
    if p.exists():
        print("Local /content/drive me ye mila (sirf local copy, Drive nahi):", os.listdir(p))
        shutil.rmtree(p)
        print("Local folder saaf kar diya.")
    from google.colab import drive
    drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/BIASCOPE")
print("BIASCOPE folder:", BASE.exists())
print("Final adapter  :", (BASE / "models/final/xlmr_lora_mbib_babe_s42").exists())
print("Temperature    :", (BASE / "models/final/final_temperature.json").exists())

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
SRC = Path("/content/drive/MyDrive/BIASCOPE/src")
SRC.mkdir(parents=True, exist_ok=True)

CORE = r'''
import bisect, html, json, os, re, string, time
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
ADAPTER_DIR = BASE_DIR / "models" / "final" / "xlmr_lora_mbib_babe_s42"
TEMP_JSON = BASE_DIR / "models" / "final" / "final_temperature.json"
MAX_LEN = 160
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEV_RE = re.compile(r"[\u0900-\u097F]")
LANG_NAME = {"en": "English", "hi": "Hindi (Devanagari script)"}

def detect_lang(text):
    dev = len(DEV_RE.findall(text))
    lat = len(re.findall(r"[A-Za-z]", text))
    return "hi" if dev > lat else "en"

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def word_scores(text, attr, offsets):
    spans = [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
    starts = [s for s, _ in spans]
    scores = np.zeros(len(spans))
    for a, (s, e) in zip(attr, offsets):
        if e <= s:
            continue
        pos = max(s, e - 1)
        k = bisect.bisect_right(starts, pos) - 1
        if k >= 0 and pos < spans[k][1]:
            scores[k] += a
    return [text[s:e] for s, e in spans], scores

def render_html(words, scores):
    mx = max(float(np.max(np.abs(scores))) if len(scores) else 0.0, 1e-9)
    parts = []
    for w, s in zip(words, scores):
        a = min(abs(float(s)) / mx, 1.0)
        color = "255,80,80" if s > 0 else "70,130,255"
        style = "background: rgba(%s,%.2f); padding:1px 3px; border-radius:3px; margin:1px;" % (color, 0.05 + 0.75 * a)
        parts.append('<span style="%s">%s</span>' % (style, html.escape(w)))
    legend = ('<div style="font-size:12px;margin-bottom:6px">'
              '<span style="background:rgba(255,80,80,.6);padding:1px 4px">red = biased ki taraf</span> '
              '<span style="background:rgba(70,130,255,.6);padding:1px 4px">blue = neutral ki taraf</span></div>')
    return legend + '<div style="font-size:17px;line-height:2.1">' + " ".join(parts) + "</div>"

class BiasModel:
    def __init__(self, adapter_dir=ADAPTER_DIR, temp_json=TEMP_JSON):
        adapter_dir = Path(adapter_dir)
        assert adapter_dir.exists(), "Adapter nahi mila: %s" % adapter_dir
        self.tok = AutoTokenizer.from_pretrained(str(adapter_dir))
        assert self.tok.is_fast, "Fast tokenizer chahiye"
        base_name = PeftConfig.from_pretrained(str(adapter_dir)).base_model_name_or_path
        base = AutoModelForSequenceClassification.from_pretrained(base_name, num_labels=2)
        self.model = PeftModel.from_pretrained(base, str(adapter_dir)).to(DEVICE).eval()
        self.emb = self.model.base_model.model.get_input_embeddings()
        self.T = 1.0
        if Path(temp_json).exists():
            self.T = float(json.load(open(temp_json))["temperature"])

    @torch.no_grad()
    def margins(self, texts, bs=32):
        texts = [str(t) for t in texts]
        if not texts:
            return np.array([])
        order = np.argsort([len(t) for t in texts])
        out = np.zeros(len(texts))
        for b in range(0, len(texts), bs):
            idx = order[b:b + bs]
            enc = self.tok([texts[i] for i in idx], return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_LEN).to(DEVICE)
            lg = self.model(**enc).logits.float()
            out[idx] = (lg[:, 1] - lg[:, 0]).cpu().numpy()
        return out

    def probs(self, texts, calibrated=True):
        return sigmoid(self.margins(texts) / (self.T if calibrated else 1.0))

    def explain(self, text, steps=32, top_k=5):
        enc = self.tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN, return_offsets_mapping=True)
        offsets = enc.pop("offset_mapping")[0].tolist()
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        ids, mask = enc["input_ids"], enc["attention_mask"]
        with torch.no_grad():
            x = self.emb(ids)
            base_ids = torch.full_like(ids, self.tok.pad_token_id)
            base_ids[0, 0], base_ids[0, -1] = self.tok.cls_token_id, self.tok.sep_token_id
            b = self.emb(base_ids)
        total = torch.zeros_like(x)
        for k in range(1, steps + 1):
            e = (b + (k / steps) * (x - b)).detach().requires_grad_(True)
            lg = self.model(inputs_embeds=e, attention_mask=mask).logits[0]
            g, = torch.autograd.grad(lg[1] - lg[0], e)
            total += g
        attr = ((x - b) * total / steps).sum(-1)[0].detach().cpu().numpy()
        words, scores = word_scores(text, attr, offsets)
        top = [(words[i], float(scores[i])) for i in np.argsort(-scores)[:top_k]]
        p_cal = float(self.probs([text])[0])
        return dict(words=words, scores=scores, top=top, p_cal=p_cal)

def get_api_key():
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get("GEMINI_API_KEY")
        except Exception:
            key = None
    if not key:
        import getpass
        key = getpass.getpass("Gemini API key (Colab Secrets me GEMINI_API_KEY rakhna behtar hai): ")
    return key

class Gemini:
    PREFERRED = ["gemini-3.5-flash", "gemini-3-flash-preview", "gemini-2.5-flash", "gemini-2.0-flash"]

    def __init__(self, api_key=None, model=None, min_interval=2.0):
        from google import genai
        from google.genai import types
        self.types = types
        self.client = genai.Client(api_key=api_key or get_api_key())
        self.min_interval = min_interval
        self._last = 0.0
        self.available = self._list_models()
        self.model = model or self._pick()

    def _list_models(self):
        names = []
        try:
            for m in self.client.models.list():
                acts = getattr(m, "supported_actions", None) or []
                if (not acts) or ("generateContent" in acts):
                    names.append(m.name.replace("models/", ""))
        except Exception as e:
            print("model list nahi mili:", repr(e)[:150])
        return names

    def _pick(self):
        for p in self.PREFERRED:
            if p in self.available:
                return p
        bad = ("lite", "image", "tts", "live", "audio", "embedding", "robotics", "computer")
        flash = [n for n in self.available if "flash" in n and not any(b in n for b in bad)]
        if flash:
            return sorted(flash)[-1]
        return self.PREFERRED[0]

    def ask(self, prompt, json_mode=True, retries=6):
        for attempt in range(retries):
            wait = self.min_interval - (time.time() - self._last)
            if wait > 0:
                time.sleep(wait)
            try:
                cfg = self.types.GenerateContentConfig(response_mime_type="application/json") if json_mode else None
                resp = self.client.models.generate_content(model=self.model, contents=prompt, config=cfg)
                self._last = time.time()
                txt = getattr(resp, "text", None)
                return txt if txt else None
            except Exception as e:
                self._last = time.time()
                s = str(e)
                transient = any(k in s for k in ("429", "500", "503", "RESOURCE_EXHAUSTED", "UNAVAILABLE", "DEADLINE", "overloaded"))
                if (not transient) or attempt == retries - 1:
                    raise
                time.sleep(min(60, 4 * 2 ** attempt))

    def ask_field(self, prompt, field):
        txt = self.ask(prompt, json_mode=True)
        if txt is None:
            return None
        t = re.sub(r"^```(?:json)?|```$", "", txt.strip(), flags=re.M).strip()
        try:
            obj = json.loads(t)
            if isinstance(obj, list):
                obj = obj[0] if obj else {}
            v = obj.get(field) if isinstance(obj, dict) else None
            return str(v).strip() if v else None
        except Exception:
            m = re.search(r'"%s"\s*:\s*"(.*?)"\s*[,}]' % re.escape(field), t, flags=re.S)
            return m.group(1).strip() if m else None

def explain_prompt(text, p, top_words, lang):
    return ("You explain the output of a media-bias classifier. The classifier, not you, decided the label.\n"
            "Text (" + LANG_NAME[lang] + "): " + json.dumps(text, ensure_ascii=False) + "\n"
            "Classifier: calibrated probability that the text is biased = " + format(p, ".2f") + "\n"
            "Words that most influenced the score (Integrated Gradients): " + ", ".join(top_words) + "\n"
            "Task: in 2-3 sentences, explain why the text reads as biased (or as mostly neutral if the probability is "
            "below 0.5), citing the influential words as evidence. Do not state facts that are not in the text. "
            "Write the explanation in " + LANG_NAME[lang] + ".\n"
            'Return JSON: {"explanation": "..."}')

def rewrite_prompt(text, lang):
    return ("Rewrite the text below in a neutral, factual tone.\n"
            "Rules: keep the same language (" + LANG_NAME[lang] + "); keep every factual claim, name and number; "
            "remove or replace loaded words, insults, sensational or clickbait phrasing, speculation or opinion "
            "presented as fact; keep the length similar; do not add new facts.\n"
            "Text: " + json.dumps(text, ensure_ascii=False) + "\n"
            'Return JSON: {"rewrite": "..."}')

def paraphrase_prompt(text, lang):
    return ("Paraphrase the text below with different wording but keep its meaning, tone and level of emotional or "
            "loaded language exactly the same. Do NOT make it more neutral. Keep the same language (" + LANG_NAME[lang] + ").\n"
            "Text: " + json.dumps(text, ensure_ascii=False) + "\n"
            'Return JSON: {"paraphrase": "..."}')
'''
(SRC / "biascope_core.py").write_text(CORE, encoding="utf-8")
print("Saved:", SRC / "biascope_core.py")

In [ ]:
# ============================================================
# BIASCOPE — Notebook 09 (v2): LLM layer — explanation + neutral rewrite
# Backends: "local" (Qwen3-8B 4-bit, GPU, deterministic, no quota) | "gemini" (API)
# Counterfactual: classifier score before/after rewrite, vs paraphrase control
# Cache Drive me (backend-wise) => disconnect/quota ke baad resume
# ============================================================
!pip -q uninstall -y torchao
!pip -q install -U google-genai sentence-transformers peft transformers accelerate bitsandbytes

import gc, importlib, json, re, sys, time, warnings
from datetime import date
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from google.colab import drive

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/BIASCOPE")
sys.path.insert(0, str(BASE_DIR / "src"))
import biascope_core as core
importlib.reload(core)

# ---------------- Config ----------------
LLM_BACKEND = "local"                      # "local" | "gemini"
LOCAL_MODEL_ID = "Qwen/Qwen3-8B"           # OOM aaye to "Qwen/Qwen3-4B"
GEMINI_MODEL = "gemini-3.1-flash-lite"     # sirf backend="gemini" ke liye
N_EN, N_HI, N_NEUTRAL = 40, 30, 15         # biased EN, biased HI(gold), neutral control (EN)
N_EXPLAIN = {"babe_en": 12, "gold_hi": 8}
SIM_THRESHOLD = 0.80                       # LaBSE cosine: meaning preserved maana jaata hai
MIN_INTERVAL = 2.5                         # gemini: calls ke beech seconds
LOCAL_BATCH = 8
IG_STEPS = 32
SEED = 42

assert LLM_BACKEND in ("local", "gemini")
if LLM_BACKEND == "local":
    assert torch.cuda.is_available(), "Local backend ko GPU chahiye (Runtime > Change runtime type > T4 GPU)."

OUT = BASE_DIR / "outputs" / "nb09_llm_layer"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / f"{LLM_BACKEND}_cache.json"          # gemini_cache.json purana wala hi hai
cache = json.load(open(CACHE)) if CACHE.exists() else {}
print("Backend:", LLM_BACKEND, "| cache entries:", len(cache))

# ---------------- Classifier ----------------
bm = core.BiasModel()
print(f"Classifier loaded | T = {bm.T:.3f} | device: {core.DEVICE}")

# ---------------- Items ----------------
from datasets import load_dataset
babe = load_dataset("mediabiasgroup/BABE")["test"].to_pandas()
babe["text"] = babe["text"].astype(str)
gold = pd.read_csv(BASE_DIR / "data" / "multilingual" / "gold" / "hindi_gold_llm_labelled.csv", encoding="utf-8-sig")
gold = gold[(gold["usable"] == 1) & (gold["confidence"] >= 2) & (gold["bias_label"] == 1)]

rng = np.random.default_rng(SEED)
def sample(df, n):
    return df.iloc[np.sort(rng.choice(len(df), min(n, len(df)), replace=False))]

items = []
for k, (_, r) in enumerate(sample(babe[babe["label"] == 1], N_EN).iterrows()):
    items.append(dict(set="babe_en", role="biased", id=str(r["uuid"]), text=r["text"], lang="en", explain=k < N_EXPLAIN["babe_en"]))
for k, (_, r) in enumerate(sample(gold, N_HI).iterrows()):
    items.append(dict(set="gold_hi", role="biased", id=str(r["gold_id"]), text=str(r["text_hi"]), lang="hi", explain=k < N_EXPLAIN["gold_hi"]))
for _, r in sample(babe[babe["label"] == 0], N_NEUTRAL).iterrows():
    items.append(dict(set="babe_en_neutral_ctrl", role="neutral", id=str(r["uuid"]), text=r["text"], lang="en", explain=False))
order = {"gold_hi": 0, "babe_en_neutral_ctrl": 1, "babe_en": 2}      # Hindi pehle (quota/time kam pade to bhi)
items.sort(key=lambda i: order[i["set"]])

p0 = bm.probs([i["text"] for i in items])
for it, p in zip(items, p0):
    it["p0"] = float(p)

# ---------------- Build LLM tasks ----------------
def K(task, it):
    return f"{task}|{it['set']}|{it['id']}"

for it in items:
    it["top_list"] = []
    if it["explain"]:
        ex = bm.explain(it["text"], steps=IG_STEPS)
        it["top_list"], it["p_cal"] = [w for w, _ in ex["top"]], ex["p_cal"]

tasks = []
for it in items:
    tasks.append((K("rewrite", it), core.rewrite_prompt(it["text"], it["lang"]), "rewrite"))
    if it["role"] == "biased":
        tasks.append((K("paraphrase", it), core.paraphrase_prompt(it["text"], it["lang"]), "paraphrase"))
    if it["explain"]:
        tasks.append((K("explain", it), core.explain_prompt(it["text"], it["p_cal"], it["top_list"], it["lang"]), "explanation"))
todo = [t for t in tasks if t[0] not in cache]
print(f"Items: {len(items)} | tasks: {len(tasks)} | cached: {len(tasks) - len(todo)} | to run: {len(todo)}")

def save_cache():
    json.dump(cache, open(CACHE, "w"), ensure_ascii=False)

def parse_field(txt, field):
    if not txt:
        return None
    t = re.sub(r"<think>.*?</think>", "", txt, flags=re.S).strip()
    t = re.sub(r"^```(?:json)?|```$", "", t, flags=re.M).strip()
    m = re.search(r"\{.*\}", t, flags=re.S)
    if m:
        try:
            obj = json.loads(m.group(0))
            v = obj.get(field) if isinstance(obj, dict) else None
            if v:
                return str(v).strip()
        except Exception:
            pass
    m = re.search(r'"%s"\s*:\s*"(.*?)"\s*[,}]' % re.escape(field), t, flags=re.S)
    return m.group(1).strip() if m else None

# ---------------- Run LLM ----------------
t0 = time.time()
if LLM_BACKEND == "gemini":
    gem = None
    for mdl in [GEMINI_MODEL, None]:
        try:
            g = core.Gemini(model=mdl, min_interval=MIN_INTERVAL)
            if g.ask_field('Return JSON: {"ok": "yes"}', "ok"):
                gem = g
                break
        except Exception as e:
            print(f"Gemini ({mdl}) fail:", repr(e)[:150])
    assert gem is not None, "Gemini ping fail — key/quota/model check karo, ya LLM_BACKEND='local' karo."
    LLM_ID = gem.model
    print("Gemini model:", LLM_ID)
    fails = 0
    for n, (key, prompt, field) in enumerate(todo, 1):
        try:
            out = gem.ask_field(prompt, field)
            fails = 0
        except Exception as e:
            fails += 1
            print(f"   ⚠ API error: {repr(e)[:140]}")
            if fails >= 3:
                print("   3 baar lagataar fail — API calls rok di (quota?). Cache save hai.")
                break
            continue
        cache[key] = out if out else "__EMPTY__"
        save_cache()
        if n % 10 == 0 or n == len(todo):
            print(f"  {n}/{len(todo)} | {(time.time() - t0) / 60:.1f} min")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    LLM_ID = LOCAL_MODEL_ID
    if todo:
        print("Loading", LOCAL_MODEL_ID, "(pehli baar ~8-16 GB download hota hai) ...")
        ltok = AutoTokenizer.from_pretrained(LOCAL_MODEL_ID)
        ltok.padding_side = "left"
        if ltok.pad_token is None:
            ltok.pad_token = ltok.eos_token
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                 bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        llm = AutoModelForCausalLM.from_pretrained(LOCAL_MODEL_ID, quantization_config=bnb, device_map="auto").eval()
        print("✓ LLM loaded | GPU mem used (GB):", round(torch.cuda.memory_allocated() / 1e9, 1))

        @torch.no_grad()
        def gen_batch(prompts, max_new_tokens=400):
            try:
                texts = [ltok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                                  add_generation_prompt=True, enable_thinking=False) for p in prompts]
                enc = ltok(texts, return_tensors="pt", padding=True).to(llm.device)
                out = llm.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                                   temperature=None, top_p=None, top_k=None, pad_token_id=ltok.pad_token_id)
                n_in = enc["input_ids"].shape[1]
                return [ltok.decode(o[n_in:], skip_special_tokens=True) for o in out]
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if len(prompts) == 1:
                    raise
                h = len(prompts) // 2
                return gen_batch(prompts[:h], max_new_tokens) + gen_batch(prompts[h:], max_new_tokens)

        todo_sorted = sorted(todo, key=lambda t: len(t[1]))
        for b in range(0, len(todo_sorted), LOCAL_BATCH):
            chunk = todo_sorted[b:b + LOCAL_BATCH]
            outs = gen_batch([c[1] for c in chunk])
            for (key, _, field), txt in zip(chunk, outs):
                v = parse_field(txt, field)
                cache[key] = v if v else "__EMPTY__"
            save_cache()
            print(f"  {min(b + LOCAL_BATCH, len(todo_sorted))}/{len(todo_sorted)} | {(time.time() - t0) / 60:.1f} min")
        del llm
        gc.collect(); torch.cuda.empty_cache()
        print("LLM memory se hata diya")

# ---------------- Assemble ----------------
def get(task, it):
    v = cache.get(K(task, it))
    return None if v in (None, "__EMPTY__") else v

for it in items:
    it["rewrite"], it["paraphrase"], it["explanation"] = get("rewrite", it), get("paraphrase", it), get("explain", it)
    it["top_words"] = ", ".join(it["top_list"])
D = pd.DataFrame([{k: v for k, v in it.items() if k not in ("top_list", "explain", "p_cal")} for it in items])
print("\nRewrites mile:", int(D["rewrite"].notna().sum()), "/", len(D), "| paraphrases:", int(D["paraphrase"].notna().sum()))

# ---------------- Scores + similarity ----------------
from sentence_transformers import SentenceTransformer
labse = SentenceTransformer("sentence-transformers/LaBSE", device=core.DEVICE)

def score_and_sim(col):
    ok = D[col].notna().values
    p, s, lr = np.full(len(D), np.nan), np.full(len(D), np.nan), np.full(len(D), np.nan)
    if ok.any():
        txt = D.loc[ok, col].astype(str).tolist()
        orig = D.loc[ok, "text"].tolist()
        p[ok] = bm.probs(txt)
        a = labse.encode(orig, normalize_embeddings=True, batch_size=32)
        b = labse.encode(txt, normalize_embeddings=True, batch_size=32)
        s[ok] = (a * b).sum(1)
        lr[ok] = [len(x.split()) / max(len(o.split()), 1) for x, o in zip(txt, orig)]
    return p, s, lr

D["p_rewrite"], D["sim_rewrite"], D["len_ratio_rewrite"] = score_and_sim("rewrite")
D["p_para"], D["sim_para"], D["len_ratio_para"] = score_and_sim("paraphrase")
D["unchanged_rewrite"] = (D["rewrite"].astype(str).str.strip() == D["text"].astype(str).str.strip()) & D["rewrite"].notna()
D.to_csv(OUT / f"nb09_rewrite_eval_{LLM_BACKEND}.csv", index=False, encoding="utf-8-sig")

# ---------------- Summary ----------------
def boot(vals, n=1000, seed=0):
    v = np.asarray(vals, float); v = v[~np.isnan(v)]
    if len(v) == 0:
        return np.nan, (np.nan, np.nan)
    r = np.random.default_rng(seed)
    m = [v[r.integers(0, len(v), len(v))].mean() for _ in range(n)]
    return float(v.mean()), tuple(np.percentile(m, [2.5, 97.5]))

rows = []
for name, d in D.groupby("set"):
    d = d[d["p_rewrite"].notna()]
    if len(d) == 0:
        continue
    drop, ci = boot(d["p0"] - d["p_rewrite"])
    row = dict(set=name, n=len(d), p0=d["p0"].mean(), p_rewrite=d["p_rewrite"].mean(),
               drop_rewrite=drop, drop_ci=f"[{ci[0]:.3f}, {ci[1]:.3f}]",
               sim_rewrite=d["sim_rewrite"].mean(), sim_ok_rate=(d["sim_rewrite"] >= SIM_THRESHOLD).mean(),
               unchanged_rate=d["unchanged_rewrite"].mean())
    fl = d[d["p0"] >= 0.5]
    if len(fl):
        row["flip_rewrite(p0>=.5)"] = (fl["p_rewrite"] < 0.5).mean()
        row["good_rewrite_rate"] = ((fl["p_rewrite"] < 0.5) & (fl["sim_rewrite"] >= SIM_THRESHOLD)).mean()
        row["n_flagged"] = len(fl)
    dp = d[d["p_para"].notna()]
    if len(dp):
        net, nci = boot(dp["p_para"] - dp["p_rewrite"])
        row.update(p_para=dp["p_para"].mean(), sim_para=dp["sim_para"].mean(),
                   net_vs_paraphrase=net, net_ci=f"[{nci[0]:.3f}, {nci[1]:.3f}]")
        fp = dp[dp["p0"] >= 0.5]
        if len(fp):
            row["flip_para(p0>=.5)"] = (fp["p_para"] < 0.5).mean()
    rows.append(row)
S = pd.DataFrame(rows).set_index("set")
print("\n" + "=" * 70 + f"\nNEUTRAL REWRITE ({LLM_ID}) — classifier score before/after (calibrated P(biased))\n" + "=" * 70)
display(S.round(3).T)
S.to_csv(OUT / f"nb09_summary_{LLM_BACKEND}.csv")
print("""
KAISE PADHNA:
- drop_rewrite > 0 aur CI zero se door: rewrite se bias score gira.
- net_vs_paraphrase > 0: rewrite ka drop paraphrase-only se ZYADA hai => asar 'neutral karne' ka hai.
- good_rewrite_rate: flagged sentences jinka score 0.5 se neeche aaya AND meaning preserved (LaBSE >= threshold).
- unchanged_rate zyada ho to LLM sirf copy kar raha hai (rewrite kamzor).
- neutral_ctrl: neutral sentences ka score rewrite se zyada nahi badalna chahiye.
- NOTE: neutrality classifier se naapi gayi hai (human check nahi) — report me yahi likho.
""")

pd.set_option("display.max_colwidth", 220)
for _, r in D[(D["role"] == "biased") & D["rewrite"].notna()].groupby("set").head(3).iterrows():
    print(f"\n[{r['set']}] P: {r['p0']:.2f} -> {r['p_rewrite']:.2f} | sim {r['sim_rewrite']:.2f}")
    print("  ORIGINAL:", r["text"])
    print("  REWRITE :", r["rewrite"])
for _, r in D[D["explanation"].notna()].groupby("set").head(2).iterrows():
    print(f"\n[EXPLANATION | {r['set']}] top words: {r['top_words']}\n  {r['explanation']}")

json.dump(dict(backend=LLM_BACKEND, llm=LLM_ID, date=str(date.today()), n_items=len(items),
               decoding="greedy (do_sample=False), 4-bit nf4" if LLM_BACKEND == "local" else "API default sampling",
               sim_threshold=SIM_THRESHOLD, classifier="xlmr_lora_mbib_babe_s42", temperature=bm.T,
               prompts=dict(explain=core.explain_prompt("<TEXT>", 0.5, ["<w>"], "en"),
                            rewrite=core.rewrite_prompt("<TEXT>", "en"),
                            paraphrase=core.paraphrase_prompt("<TEXT>", "en"))),
          open(OUT / f"llm_config_{LLM_BACKEND}.json", "w"), indent=2, ensure_ascii=False)
print("\nSaved in:", OUT)